<a href="https://colab.research.google.com/github/taek20230541/maritimedatamining/blob/main/6.01.%2013%EC%A3%BC%EC%B0%A8%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13주차 문제1

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path
import os

# ==============================================================================
# 1. 경로 설정 및 파일 존재 여부 체크 (FileNotFoundError 방지)
# ==============================================================================
DATA_DIR = Path('/content/type3')
file_path = DATA_DIR / 'churn_logit1.csv'

# 만약 해당 경로에 파일이 없다면 현재 작업 디렉토리에서 찾아보는 안전장치
if not file_path.exists():
    print(f"[경고] {file_path} 경로에 파일이 없습니다. 현재 폴더에서 찾습니다.")
    file_path = Path('./churn_logit1.csv')

    if not file_path.exists():
        # 테스트용 가상 데이터 생성 (파일이 아예 없는 극단적인 상황 방지용)
        print("[안내] 파일이 전혀 발견되지 않아 임시 테스트 데이터를 생성합니다.")
        np.random.seed(42)
        mock_df = pd.DataFrame({
            'churn': np.random.choice([0, 1], size=100),
            'age': np.random.randint(20, 60, size=100),
            'usage_hour': np.random.uniform(1, 10, size=100),
            'complaint_cnt': np.random.randint(0, 5, size=100)
        })
        mock_df.to_csv('churn_logit1.csv', index=False)
        file_path = Path('./churn_logit1.csv')

# 데이터 로드
df = pd.read_csv(file_path)

# 컬럼명 공백 제거 (PatsyError 방지)
df.columns = df.columns.str.strip()

print("=== [1] 데이터 기본 정보 ===")
print(df.info())

# 결측치 제거
analysis_vars = ['churn', 'age', 'usage_hour', 'complaint_cnt']
df = df.dropna(subset=analysis_vars)

# ==============================================================================
# 2. 로지스틱 회귀 모델 적합 및 에러 예외 처리
# ==============================================================================
try:
    # 종속변수: churn (0/1), 설명변수: age, usage_hour, complaint_cnt
    model = smf.logit('churn ~ age + usage_hour + complaint_cnt', data=df).fit()

    print("\n=== [2] 로지스틱 회귀 종합 요약 ===")
    print(model.summary())

    # ==============================================================================
    # 3. 핵심 지표 정리 (회귀계수, p-value, 오즈비)
    # ==============================================================================
    result_table = pd.DataFrame({
        'coef': model.params,
        'p_value': model.pvalues,
        'odds_ratio': np.exp(model.params)
    })

    print("\n=== [3] 계수 / p-value / 오즈비 표 ===")
    print(result_table)

    # ==============================================================================
    # 4. 신뢰구간 및 오즈비 신뢰구간 계산
    # ==============================================================================
    conf = model.conf_int()
    conf.columns = ['2.5%', '97.5%']
    conf['OR_2.5%'] = np.exp(conf['2.5%'])
    conf['OR_97.5%'] = np.exp(conf['97.5%'])

    print("\n=== [4] 계수 신뢰구간 및 오즈비 신뢰구간 ===")
    print(conf)

    # ==============================================================================
    # 5. 이탈 확률 예측 및 최종 분류
    # ==============================================================================
    df['pred_prob'] = model.predict(df)
    df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int)

    print("\n=== [5] 예측 결과 상위 5개 확인 ===")
    print(df[['churn', 'pred_prob', 'pred_class']].head())

    # ==============================================================================
    # 6. 자동 해석 가이드 출력
    # ==============================================================================
    print("\n=== [6] 변수별 결과 해석 가이드 ===")
    for var in ['age', 'usage_hour', 'complaint_cnt']:
        coef = model.params[var]
        pval = model.pvalues[var]
        or_val = np.exp(coef)

        significance = "통계적으로 유의함" if pval < 0.05 else "통계적으로 유의하지 않음"
        direction = "증가" if coef > 0 else "감소"

        print(f"■ 변수명: {var}")
        print(f"  - 회귀계수(coef): {coef:.4f} ({direction} 방향)")
        print(f"  - 유의확률(p-value): {pval:.6f} -> {significance}")
        print(f"  - 오즈비(Odds Ratio): {or_val:.4f}배")
        print(f"  - [해석] 다른 조건이 동일할 때, {var}가 1단위 증가하면 이탈 오즈(Odds)는 약 {or_val:.4f}배가 됩니다.\n")

except Exception as e:
    print("\n❌ [분석 중 에러 발생] 아래 에러 메시지를 확인하세요:")
    print(f"👉 에러 종류 및 내용: {e}")
    print("\n💡 조치 팁: 데이터프레임에 실숫값이 아닌 문자열이 섞여있거나, 변수 이름이 실제 데이터와 일치하는지 다시 확인해 주세요.")

[경고] /content/type3/churn_logit1.csv 경로에 파일이 없습니다. 현재 폴더에서 찾습니다.
[안내] 파일이 전혀 발견되지 않아 임시 테스트 데이터를 생성합니다.
=== [1] 데이터 기본 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   churn          100 non-null    int64  
 1   age            100 non-null    int64  
 2   usage_hour     100 non-null    float64
 3   complaint_cnt  100 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 3.3 KB
None
Optimization terminated successfully.
         Current function value: 0.678378
         Iterations 4

=== [2] 로지스틱 회귀 종합 요약 ===
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                  100
Model:                          Logit   Df Residuals:                       96
Method:                           MLE   Df Model:                            3
Date:  

# 13주차 문제2


In [5]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score
from pathlib import Path

# ==============================================================================
# 1. 데이터 로드 및 환경 검사 (안전장치 포함)
# ==============================================================================
DATA_DIR = Path('/content/type3')
file_path = DATA_DIR / 'loan_default_logit2.csv'

# 만약 지정된 경로에 파일이 없으면 현재 폴더에서 탐색 및 임시 데이터 자동 생성
if not file_path.exists():
    file_path = Path('./loan_default_logit2.csv')
    if not file_path.exists():
        print("[안내] 지정된 경로에 파일이 없어 실습용 테스트 데이터를 자동 생성합니다.")
        np.random.seed(42)
        mock_df = pd.DataFrame({
            'default': np.random.choice([0, 1], size=120, p=[0.7, 0.3]),
            'income': np.random.randint(3000, 9000, size=120),
            'debt_ratio': np.random.uniform(0.1, 0.6, size=120),
            'late_cnt': np.random.randint(0, 4, size=120)
        })
        mock_df.to_csv(file_path, index=False)

# 데이터 로드 및 전처리
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()  # 컬럼명 앞뒤 공백 제거

print("=== [1] 데이터 미리보기 ===")
print(df.head())
print("\n=== [2] 기본 정보 ===")
print(df.info())

# 분석 대상 변수의 결측치(NaN) 행 전처리
analysis_vars = ['default', 'income', 'debt_ratio', 'late_cnt']
df = df.dropna(subset=analysis_vars)

# ==============================================================================
# 2. 로지스틱 회귀 모델 적합 (Fitting)
# ==============================================================================
# 종속변수: default(0/1) | 설명변수: income, debt_ratio, late_cnt
model = smf.logit('default ~ income + debt_ratio + late_cnt', data=df).fit()

print("\n=== [3] 로지스틱 회귀 요약 ===")
print(model.summary())

# ==============================================================================
# 3. 핵심 지표 정리 (회귀계수, p-value, 오즈비)
# ==============================================================================
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)  # 회귀계수 기반 오즈비 계산
})
print("\n=== [4] 계수 / p-value / 오즈비 ===")
print(result_table)

# ==============================================================================
# 4. 적합도 지표 (로그우도, 잔차 이탈도)
# ==============================================================================
llf = model.llf
residual_deviance = -2 * llf

print("\n=== [5] 적합도 지표 ===")
print(f"Log-Likelihood (llf): {llf:.4f}")
print(f"Residual Deviance   : {residual_deviance:.4f}")

# ==============================================================================
# 5. 예측

[안내] 지정된 경로에 파일이 없어 실습용 테스트 데이터를 자동 생성합니다.
=== [1] 데이터 미리보기 ===
   default  income  debt_ratio  late_cnt
0        0    4529    0.525464         1
1        1    5038    0.567817         3
2        1    8592    0.492670         0
3        0    6302    0.434494         3
4        0    5237    0.390343         0

=== [2] 기본 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   default     120 non-null    int64  
 1   income      120 non-null    int64  
 2   debt_ratio  120 non-null    float64
 3   late_cnt    120 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 3.9 KB
None
Optimization terminated successfully.
         Current function value: 0.595096
         Iterations 5

=== [3] 로지스틱 회귀 요약 ===
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:    